# FINet full project recovery and sequential training
Use this after the verified initial setup has backed up the source ZIP, data, split, smoke evidence and baseline run in MyDrive/FINet. Select a free T4 GPU and run cells in order. This restores the existing experiment; it does not create a new initialization or change the training budget.

The runner finishes original FINet, verifies/trains the matched boundary model, then evaluates both full-budget checkpoints on audited unseen data. Free runtime limits can interrupt execution; checkpoints support recovery. Actual results are written only after real computation. Final Gradio integration must still be verified with the resulting models.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, shutil, hashlib, subprocess
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/FINet')
RUN_DIR = DRIVE_ROOT / 'experiments/baseline_seed123456_b4_v1'
PROJECT_RUN = DRIVE_ROOT / 'projects/finet_matched_v1'
assert (RUN_DIR/'config.json').is_file(), 'Restore the existing baseline experiment first.'


In [ ]:
archive = Path('/content/finet_colab_training.zip')
shutil.copy2(DRIVE_ROOT/'code/finet_colab_training.zip', archive)
verification = json.loads((DRIVE_ROOT/'code/backup_verification.json').read_text())
assert hashlib.sha256(archive.read_bytes()).hexdigest() == verification['sha256'], 'Source backup integrity mismatch.'
PROJECT = Path('/content/boundary-aware-finet')
with zipfile.ZipFile(archive) as zipped:
    assert zipped.testzip() is None
    for name in zipped.namelist():
        assert not name.startswith('/') and '..' not in Path(name).parts
    zipped.extractall(PROJECT)
os.chdir(PROJECT)
assert (PROJECT/'scripts/run_colab_project.py').is_file(), 'Use the current project ZIP.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-training.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
import torch
assert torch.cuda.is_available(), 'A real CUDA GPU is required.'
print({'gpu':torch.cuda.get_device_name(0), 'free_total_bytes':torch.cuda.mem_get_info(), 'torch':torch.__version__})


In [ ]:
DATA_ROOT = Path('/content/finet_data/TrainDataset')
SPLIT = DRIVE_ROOT/'splits/cod_camo_seed123456.json'
DATA_ARCHIVE = DRIVE_ROOT/'data/COD-TrainDataset.zip'
local_archive = Path('/content/COD-TrainDataset.zip')
if not local_archive.is_file(): shutil.copy2(DATA_ARCHIVE, local_archive)
if not (DATA_ROOT/'Imgs').is_dir():
    subprocess.run([sys.executable, 'scripts/prepare_official_data.py', '--archive', str(local_archive), '--data-root', str(DATA_ROOT), '--manifest', str(SPLIT)], check=True)
from src.training_data import file_hash, load_manifest
assert file_hash(local_archive) == '255146828426e144c54867879824bcef27075b9e2386899fa0220f77551dc7be'
manifest = load_manifest(SPLIT, DATA_ROOT)
assert manifest['fingerprint'] == '2100aa40c52f01442d59b28973fee653eec655e370c8b1c1ff3c8d40cf61d724'
FULL_CONFIG = PROJECT/'outputs/colab_full_config.json'
FULL_CONFIG.parent.mkdir(parents=True, exist_ok=True)
FULL_CONFIG.write_text((RUN_DIR/'config.json').read_text())
REPORT = DRIVE_ROOT/'experiments/baseline_seed123456_b4_v1_smoke_v1/smoke_report.json'
from src.training import TrainingConfig, run_signature, environment_report
cfg = json.loads(FULL_CONFIG.read_text())
proof = json.loads(REPORT.read_text())
assert proof['run_signature'] == run_signature(TrainingConfig(**cfg), manifest)
assert proof['hardware']['gpu'] == torch.cuda.get_device_name(0), 'GPU changed: measure new matching smoke before proceeding.'
assert proof['hardware']['environment']['source_sha256'] == environment_report()['source_sha256'], 'Training source differs from verified smoke.'
print(json.dumps(cfg, indent=2))
print('Shared initial model:', proof['initial_model_sha256'])


In [ ]:
RUN_PROJECT = False  # Enable only after reviewing the actual GPU/configuration above.
if RUN_PROJECT:
    subprocess.run([sys.executable, '-u', 'scripts/run_colab_project.py', '--config', str(FULL_CONFIG), '--baseline-smoke', str(REPORT), '--output', str(PROJECT_RUN), '--run-project'], check=True, cwd=PROJECT)
else:
    print('Long stages disabled. Set RUN_PROJECT=True after checking the evidence.')


## Recovery and deliverables
Keep the foreground cell running while it performs actual training. Do not run a second baseline launcher. This does not bypass Colab limits. After a disconnect, reconnect and rerun this notebook using the same Drive paths.

Read MyDrive/FINet/projects/finet_matched_v1/project_status.json and logs/. Full-budget states and validation-selected states are exported under models/. Successful real evaluation creates per-image CSVs, sample masks, comparison PNG/PDF files and MEASURED_RESULTS.md. A training-complete status does not prove that the final application or submission material has been verified. Partial checkpoints and validation observations must be labelled honestly.